# 11 WLASL2000 Webcam Real-Time Inference

## What this notebook does

This notebook runs the deployed WLASL2000 model on your webcam.

Pipeline:

```text
webcam frame
→ MediaPipe keypoints
→ rolling 60-frame buffer
→ model prediction every 15 frames
→ stability check across recent predictions
→ accepted / uncertain / repeat decision
```

Press **Q** to quit the camera window. Press **C** to clear the accepted transcript.

# 1. Import libraries

In [ ]:
from pathlib import Path
from collections import Counter, deque
import json, time, warnings

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

# 2. Load deployment config

In [ ]:
PROJECT_ROOT = Path("E:/Be_My_Ear")

DEPLOY_DIR = PROJECT_ROOT / "app" / "models" / "ASL" / "WLASL2000"
CONFIG_FILE = DEPLOY_DIR / "wlasl2000_deployment_config.json"

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    config = json.load(f)

MODEL_PATH = Path(config["model_path"])
LABEL_MAP_PATH = Path(config["label_map_path"])

with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    raw_label_map = json.load(f)

id_to_gloss = {int(k): v["gloss"] for k, v in raw_label_map.items()}
rules = config["confidence_rules"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Config:", CONFIG_FILE)
print("Model:", MODEL_PATH)
print("Label map:", LABEL_MAP_PATH)
print("Device:", device)
print("Confidence rules:")
print(json.dumps(rules, indent=4))

# 3. Load normalisation stats

In [ ]:
def find_norm_stats_file():
    candidates = []

    if "norm_stats_path" in config:
        candidates.append(Path(config["norm_stats_path"]))

    selected_name = config.get("selected_model_name", "").lower()
    model_dir = PROJECT_ROOT / "models" / "ASL" / "WLASL2000"

    if "light v3" in selected_name:
        candidates.append(model_dir / "wlasl2000_light_v3_two_stage_finetuned_from_wlasl1000_train_norm_stats.npz")

    if "light v2" in selected_name:
        candidates.append(model_dir / "wlasl2000_light_v2_finetuned_from_wlasl1000_train_norm_stats.npz")

    candidates.extend(sorted(model_dir.glob("*norm_stats*.npz")))

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError("Could not find WLASL2000 norm stats .npz file in models/ASL/WLASL2000.")

NORM_STATS_FILE = find_norm_stats_file()
stats = np.load(NORM_STATS_FILE)
train_mean = stats["mean"].astype(np.float32)
train_std = stats["std"].astype(np.float32)

print("Norm stats:", NORM_STATS_FILE)
print("Mean:", train_mean.shape, "Std:", train_std.shape)

# 4. Load deployed model

In [ ]:
class BiGRUAttentionDeploy(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2, dropout=0.35):
        super().__init__()

        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        bi_hidden = hidden_size * 2

        self.attention = nn.Sequential(
            nn.Linear(bi_hidden, hidden_size),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(bi_hidden, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        x = self.input_projection(x)
        gru_out, _ = self.gru(x)
        scores = self.attention(gru_out).squeeze(-1)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        context = torch.sum(gru_out * weights, dim=1)
        return self.classifier(context)


checkpoint = torch.load(MODEL_PATH, map_location=device)

num_classes = int(config["num_classes"])
input_size = int(config["input_shape"][1])
hidden_size = int(checkpoint.get("hidden_size", 320))
num_layers = int(checkpoint.get("num_layers", 2))
dropout = float(checkpoint.get("dropout", 0.35))

model = BiGRUAttentionDeploy(input_size, hidden_size, num_classes, num_layers, dropout).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded:", config["selected_model_name"])
print("Architecture:", checkpoint.get("architecture", "BiGRUAttentionDeploy"))
print("Classes:", num_classes)

# 5. Setup MediaPipe

In [ ]:
mp_holistic = mp.solutions.holistic

SEQUENCE_LENGTH = int(config["sequence_length"])
BASE_FEATURE_SIZE = int(config["base_keypoint_shape"][1])
INPUT_SIZE = int(config["input_shape"][1])

LEFT_HAND_SIZE = 21 * 3
RIGHT_HAND_SIZE = 21 * 3
POSE_SIZE = 33 * 4
FEATURE_SIZE = LEFT_HAND_SIZE + RIGHT_HAND_SIZE + POSE_SIZE

def extract_landmarks_from_results(results):
    left = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark], dtype=np.float32).flatten() if results.left_hand_landmarks else np.zeros(LEFT_HAND_SIZE, dtype=np.float32)
    right = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark], dtype=np.float32).flatten() if results.right_hand_landmarks else np.zeros(RIGHT_HAND_SIZE, dtype=np.float32)
    pose = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark], dtype=np.float32).flatten() if results.pose_landmarks else np.zeros(POSE_SIZE, dtype=np.float32)
    return np.concatenate([left, right, pose]).astype(np.float32)

def process_frame_to_keypoints(frame_bgr, holistic):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    rgb.flags.writeable = False
    results = holistic.process(rgb)
    return extract_landmarks_from_results(results)

print("MediaPipe ready.")
print("Sequence length:", SEQUENCE_LENGTH)
print("Base feature size:", BASE_FEATURE_SIZE)
print("Model input size:", INPUT_SIZE)

# 6. Prediction and decision functions

In [ ]:
def prepare_model_input(keypoint_sequence):
    keypoint_sequence = np.asarray(keypoint_sequence, dtype=np.float32)

    if keypoint_sequence.shape != (SEQUENCE_LENGTH, BASE_FEATURE_SIZE):
        raise ValueError(f"Expected {(SEQUENCE_LENGTH, BASE_FEATURE_SIZE)}, got {keypoint_sequence.shape}")

    normalised = (keypoint_sequence - train_mean.reshape(1, -1)) / (train_std.reshape(1, -1) + 1e-6)

    velocity = np.zeros_like(normalised, dtype=np.float32)
    velocity[1:] = normalised[1:] - normalised[:-1]

    features = np.concatenate([normalised, velocity], axis=1).astype(np.float32)
    return torch.tensor(features, dtype=torch.float32).unsqueeze(0)

def predict_keypoint_sequence(keypoint_sequence, top_k=5):
    x = prepare_model_input(keypoint_sequence).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)[0].detach().cpu().numpy()

    top_ids = np.argsort(probs)[-top_k:][::-1]

    top_predictions = [
        {
            "label_id": int(label_id),
            "gloss": id_to_gloss.get(int(label_id), str(label_id)),
            "probability": float(probs[label_id])
        }
        for label_id in top_ids
    ]

    top1 = top_predictions[0]
    top2_prob = top_predictions[1]["probability"] if len(top_predictions) > 1 else 0.0

    return {
        "top1_label_id": top1["label_id"],
        "top1_gloss": top1["gloss"],
        "top1_confidence": top1["probability"],
        "top1_top2_margin": float(top1["probability"] - top2_prob),
        "top_k": top_predictions
    }

def decision_from_prediction(prediction, recent_predictions=None):
    confidence = prediction["top1_confidence"]
    margin = prediction["top1_top2_margin"]

    auto_accept_confidence = float(rules.get("auto_accept_confidence", 0.45))
    uncertain_min_confidence = float(rules.get("uncertain_min_confidence", 0.25))
    required_margin = float(rules.get("top1_top2_margin", 0.05))
    min_repeated = int(rules.get("min_repeated_predictions", 2))
    stability_count = int(rules.get("stability_window_count", 3))

    stable = False

    if recent_predictions is not None and len(recent_predictions) >= min_repeated:
        last_items = list(recent_predictions)[-stability_count:]
        glosses = [item["top1_gloss"] for item in last_items]
        stable = Counter(glosses)[prediction["top1_gloss"]] >= min_repeated
    else:
        stable = True

    if confidence >= auto_accept_confidence and margin >= required_margin and stable:
        return {
            "status": "accepted",
            "stable": stable,
            "message": f"Detected sign: {prediction['top1_gloss']}",
            "top1_gloss": prediction["top1_gloss"],
            "confidence": confidence,
            "margin": margin
        }

    if confidence >= uncertain_min_confidence:
        alternatives = ", ".join([x["gloss"] for x in prediction["top_k"]])
        return {
            "status": "uncertain",
            "stable": stable,
            "message": f"I think this may mean: {prediction['top1_gloss']} | Alternatives: {alternatives}",
            "top1_gloss": prediction["top1_gloss"],
            "confidence": confidence,
            "margin": margin
        }

    return {
        "status": "repeat",
        "stable": stable,
        "message": "I am not sure. Please sign again slowly.",
        "top1_gloss": prediction["top1_gloss"],
        "confidence": confidence,
        "margin": margin
    }

# 7. Webcam settings

In [ ]:
CAMERA_INDEX = 0
PREDICT_EVERY_N_FRAMES = 15
MIN_SECONDS_BETWEEN_ACCEPTED_SIGNS = 1.5
SHOW_TOP5_ON_SCREEN = True

print("Camera index:", CAMERA_INDEX)
print("Predict every N frames:", PREDICT_EVERY_N_FRAMES)

# 8. Display helpers

In [ ]:
def draw_text_block(frame, lines, x=20, y=40, line_height=30):
    for i, line in enumerate(lines):
        cv2.putText(
            frame,
            line,
            (x, y + i * line_height),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

def format_prediction_lines(prediction, decision, accepted_signs):
    lines = [
        f"Status: {decision['status'].upper()}",
        f"Top-1: {prediction['top1_gloss']} ({prediction['top1_confidence']:.2f})",
        f"Stable: {decision['stable']} | Margin: {prediction['top1_top2_margin']:.2f}"
    ]

    if SHOW_TOP5_ON_SCREEN:
        top5 = ", ".join([x["gloss"] for x in prediction["top_k"]])
        lines.append(f"Top-5: {top5}")

    if accepted_signs:
        lines.append("Accepted: " + " ".join(accepted_signs[-6:]))

    lines.append("Q=quit | C=clear")
    return lines

def should_accept_new_sign(gloss, accepted_signs, last_accept_time):
    now = time.time()

    if accepted_signs and accepted_signs[-1] == gloss:
        return False

    if now - last_accept_time < MIN_SECONDS_BETWEEN_ACCEPTED_SIGNS:
        return False

    return True

# 9. Start webcam inference

In [ ]:
cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    raise RuntimeError(f"Could not open camera index {CAMERA_INDEX}")

keypoint_buffer = deque(maxlen=SEQUENCE_LENGTH)
recent_predictions = deque(maxlen=int(rules.get("stability_window_count", 3)))

accepted_signs = []
last_accept_time = 0.0
frame_counter = 0

latest_prediction = None
latest_decision = None

print("Starting webcam inference.")
print("Press Q in the camera window to quit.")
print("Press C in the camera window to clear transcript.")

with mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
    enable_segmentation=False,
    refine_face_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as holistic:

    while True:
        success, frame = cap.read()

        if not success:
            print("Failed to read webcam frame.")
            break

        frame = cv2.flip(frame, 1)
        keypoints = process_frame_to_keypoints(frame, holistic)
        keypoint_buffer.append(keypoints)

        frame_counter += 1

        if len(keypoint_buffer) == SEQUENCE_LENGTH and frame_counter % PREDICT_EVERY_N_FRAMES == 0:
            sequence = np.array(keypoint_buffer, dtype=np.float32)

            latest_prediction = predict_keypoint_sequence(
                sequence,
                top_k=int(rules.get("output_top_k", 5))
            )

            recent_predictions.append(latest_prediction)

            latest_decision = decision_from_prediction(
                latest_prediction,
                recent_predictions=recent_predictions
            )

            if latest_decision["status"] == "accepted":
                gloss = latest_prediction["top1_gloss"]

                if should_accept_new_sign(gloss, accepted_signs, last_accept_time):
                    accepted_signs.append(gloss)
                    last_accept_time = time.time()

        display_frame = frame.copy()

        if latest_prediction is not None and latest_decision is not None:
            lines = format_prediction_lines(latest_prediction, latest_decision, accepted_signs)
        else:
            lines = [
                f"Collecting frames: {len(keypoint_buffer)}/{SEQUENCE_LENGTH}",
                "Q=quit | C=clear"
            ]

        draw_text_block(display_frame, lines)
        cv2.imshow("Be My Ear - WLASL2000 Webcam Inference", display_frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

        if key == ord("c"):
            accepted_signs = []
            print("Transcript cleared.")

cap.release()
cv2.destroyAllWindows()

print("Webcam inference stopped.")
print("Accepted signs:", accepted_signs)
print("Auto-understanding output:", " ".join(accepted_signs))

# 10. Save webcam session summary

In [ ]:
REPORT_DIR = PROJECT_ROOT / "reports" / "inference"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = time.strftime("%Y%m%d_%H%M%S")
summary_file = REPORT_DIR / f"webcam_session_{timestamp}_wlasl2000_summary.json"

summary = {
    "timestamp": timestamp,
    "model": config["selected_model_name"],
    "accepted_signs": accepted_signs,
    "auto_understanding_output": " ".join(accepted_signs),
    "rules": rules,
    "camera_index": CAMERA_INDEX,
    "predict_every_n_frames": PREDICT_EVERY_N_FRAMES
}

with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

print("Saved summary:", summary_file)